# Users Silver
---

* Cada usuário aparece **uma única vez**
* Tipos devem seguir o documento de padronização da Silver
* Datas devem seguir os tipos especificados no documento
* Dados ruins são tratados, não ignorados

---
**Bibliotecas**

* `functions (F)` &rarr; transformações declarativas (Spark SQL style)
* `Window` &rarr; deduplicação correta

---
**Leitura da Bronze**

* Estamos confiando que a Bronze:
  * já tem schema
  * já tem metadados
  * já é rastreável

---
**Seleção de colunas**

Esse é um processo simples, mas `arquiteturalmente crítico`

* Não aplicar `select("*")`:
  * Campos inesperados quebram os contratos
  * Evolução de schema fica caótica
  * Times downstream perdem confiança

> *Aqui será definido tudo de forma explícita.*

---
**Aplicação da Tipagem nas colunas**

* Aplicar as regras que estão no **data contract**
  * Não é necessário definir a tipagem a ser aplicada depois para todas as colunas, apenas para as colunas que precisam dos ajustes necessários

---
**Normalização**

Aqui será aplicado as regras definidas no documento para a normalização das nomenclaturas dos países.

* Segue a nomenclatura com 2 dígitos
* Aplicação de `upper()` 

---
**Validações Mínimas de Qualidade**

Remover registros nullos da coluna `user_id`

---
**Deduplicação**

> *Para cada `user_id`, manter o registro com `created_at` mais recente.*

Aqui é onde escolhemos a "verdadeira" versão de um mesmo registro quando ele aparece mais de uma vez.

**Não usar `dropDuplicates()` aqui, pois não resolve a regra.**


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

* **Leitura do dados na bronze**

In [0]:
users_bronze = "/Volumes/main/lakehouse_marketing/bronze/users/"

df_bronze = spark.read\
                .format("delta")\
                .load(users_bronze)

display(df_bronze.limit(5))
display(df_bronze.printSchema())

user_id,email,country,signup_date,created_at,ingestion_timestamp,source_file
bdd640fb-0667-4ad1-9c80-317fa3b1799d,john21@example.net,BR,2025-06-23,2000-07-16T07:10:59.304001,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
07a0ca6e-0822-48f3-ac03-1199972a8469,robinsonwilliam@example.org,br,2025-02-12,2010-02-04T15:22:56.895920,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
386ecbe0-6b65-46a4-8b81-48f6b38a088c,zlawrence@example.org,null,2025-07-06,1978-12-10T11:42:31.943111,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
27cd8130-4722-4389-971a-a8766c307511,susanrogers@example.org,null,2024-10-02,1990-02-07T03:37:28.052331,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
ce9ff57f-43b7-43a6-9a8d-ca03580d7b71,blairamanda@example.com,BR,2024-02-25,1986-06-02T07:56:45.634285,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv


root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



* **Aqui faremos:**

  * *Seleção das Colunas*
  * *Definição da Tipagem*
  * *Normalização para Country*
  * *Validações Mínimas de qualidade*


In [0]:
#########################################################
#  Seleção explícita das colunas que serão usadas
#########################################################
df = df_bronze.select(
    "user_id",
    "email",
    "country",
    "signup_date",
    "created_at",
    "ingestion_timestamp",
    "source_file"
)

#############################################################
# Isso é apenas uma forma declarativa para se aplicar depois
#############################################################
df_typed = df\
            .withColumn("email", F.lower("email"))\
            .withColumn("country", F.upper("country"))\
            .withColumn("signup_date", F.to_date("signup_date"))

#####################################
#Nomalização para a coluna country
#####################################
df_normalized = df_typed.withColumn(
    "country",
    F.when(F.col("country").isin("BRA", "BRAZIL"), "BR")
     .when(F.col("country").isin("USA", "US", "UNITED STATES"), "US")
     .otherwise("UNKNOWN")
)
display(df_normalized.groupBy("country").count())
display(df_normalized.sample(0.001))


country,count
US,1709
BR,786
UNKNOWN,2505


user_id,email,country,signup_date,created_at,ingestion_timestamp,source_file
ca226cfe-4b4f-4f74-8914-741cdabc47ce,markbrown@example.com,US,2025-09-28,2017-09-06T15:42:53.951797,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00001-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-406-1-c000.csv
b575dd69-0b61-49c4-844a-a5c98e56dda6,christensenkendra@example.org,US,2024-12-13,2012-11-14T10:52:37.887280,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00003-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-407-1-c000.csv
4c61d505-3abe-4b28-83f5-9143a52a0d8d,matthewstanley@example.org,UNKNOWN,2025-05-28,2003-05-21T15:24:14.363644,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00005-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-412-1-c000.csv
b56e2a21-0dbe-492f-8855-d3342589f9f3,lisa10@example.org,UNKNOWN,2024-01-14,2005-03-08T05:23:25.687678,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00006-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-413-1-c000.csv
82aaf3a7-07cb-40ec-bcb2-34e253497e83,destinyroberts@example.net,UNKNOWN,2024-07-29,1983-04-17T00:21:46.798230,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00006-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-413-1-c000.csv


* Trazer dados onde `user_id` não é nullo

In [0]:
####################################
# Validações mínimas de qualidade
####################################
df_valid = df_normalized.filter(F.col("user_id").isNotNull())
display(df.sample(0.001))

user_id,email,country,signup_date,created_at,ingestion_timestamp,source_file
c5cc0fb0-e1a6-4208-a1f4-06277e83e815,rachelwalter@example.org,br,2024-04-29,1993-06-11T02:04:06.899229,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00002-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-411-1-c000.csv
07a76739-609e-4d33-ad92-abbf65c51c9a,jimenezmaria@example.com,null,2024-05-05,1970-09-29T09:21:49.295090,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00002-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-411-1-c000.csv
04a6dca6-f908-417b-a66b-b9324f1da23b,cynthiaballard@example.net,BR,2024-01-07,1990-04-30T18:27:53.415478,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00004-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-408-1-c000.csv
eb08fc43-80dd-4f64-8acc-2c22849b25b6,david48@example.org,null,2024-05-27,2012-02-25T15:18:38.841762,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00004-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-408-1-c000.csv
4a78ffe0-ef60-4bc6-b0f1-da53b5e8bbd4,mejiadaniel@example.com,br,2024-08-26,2000-11-17T10:31:11.421812,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00004-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-408-1-c000.csv
662d826f-86e8-4510-a192-abeccb685168,fhatfield@example.org,br,2025-11-07,1988-04-09T16:23:18.677758,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00005-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-412-1-c000.csv
fac1bde9-0641-42bb-ade1-266ba56f50b4,johnsonnathan@example.net,null,2024-05-21,1974-05-21T00:37:57.343311,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00006-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-413-1-c000.csv


* Deduplicação
---

A regra: *Aplicar deduplicação em `user_id` com `created_at` mais recente*.

In [0]:
# Primeiro particionamos por user_id e ordenamos por signup_date
window = Window.partitionBy("user_id").orderBy(F.col("signup_date").desc())

# Após definimos a janela, o row_number define um rank para cada registro
# como ordenamos de forma decrescente, o maior "signup_date" de cada "user_id"
# terá o rank 1, com isso, conseguimos filtrar apenas o que for igual a  1.
df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print(f"Sem deduplicação: {df_valid.count()}")
print(f"Com deduplicação: {df_dedup.count()}")

Sem deduplicação: 4771
Com deduplicação: 4771


#### Escrita na SILVER

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS main.lakehouse_marketing.silver;

In [0]:
silver_path = "/Volumes/main/lakehouse_marketing/silver/users/"

df_dedup.write\
        .format("delta")\
        .mode("overwrite")\
        .save(silver_path)

* Validação

In [0]:
spark.read.format("delta").load(silver_path).display()

user_id,email,country,signup_date,created_at,ingestion_timestamp,source_file
000616d1-4715-40e4-935f-6c60ecc06aeb,whorton@example.com,UNKNOWN,2025-02-08,1979-03-15T06:54:09.710521,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00003-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-407-1-c000.csv
000df764-9d46-4a9a-8a1d-9238fcf9abbb,kristineprince@example.com,UNKNOWN,2024-03-17,2003-03-25T02:38:36.923263,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00002-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-411-1-c000.csv
002d111d-08aa-49da-94c5-be4f63e947bf,julie89@example.com,US,2025-03-12,2019-03-14T06:16:11.346119,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00006-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-413-1-c000.csv
00595cd2-6a00-4647-9c74-e1da80989251,laura67@example.org,US,2024-08-02,2015-10-12T18:44:41.000954,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00005-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-412-1-c000.csv
005e0b17-dbae-47c4-b4d2-c8b83facb825,nathaniel41@example.org,US,2024-10-09,1994-05-30T06:37:48.556390,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00005-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-412-1-c000.csv
006e0070-bca6-46b0-93b0-d27e61547769,bryan83@example.com,UNKNOWN,2025-12-15,1987-11-25T14:38:02.826392,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00001-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-406-1-c000.csv
0075d3d5-ba94-4a66-ad7e-4a452099e71f,xrosales@example.net,UNKNOWN,2024-01-22,2003-09-25T13:32:42.499544,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00000-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-410-1-c000.csv
008f3fc8-e82f-48b8-8e86-3135a418587a,michael87@example.com,US,2025-03-18,2024-03-11T10:46:14.166969,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00007-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-409-1-c000.csv
00a53ee7-3fc3-4d4c-b6ae-71ce2591855b,ohall@example.com,UNKNOWN,2025-05-08,2014-03-07T09:04:02.399296,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00004-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-408-1-c000.csv
00aef0aa-2f13-4be4-a63b-88605953d9c0,eanderson@example.net,BR,2024-11-22,1971-04-06T07:23:10.304799,2026-01-30T16:07:16.986Z,dbfs:/Volumes/main/lakehouse_marketing/raw/users/part-00005-tid-2823656804105638587-087b001a-0ece-4a19-92d8-1542074bf073-412-1-c000.csv


In [0]:
spark.read.format("delta").load(silver_path).count()

4771